In [1]:
"""
Enhanced Grazioso Salvare Rescue Candidate Dashboard.

This dashboard uses the reusable data preparation, intake-to-outcome
matching, and rescue suitability scoring modules created for the
CS 499 enhancement.
"""

from __future__ import annotations

import base64
import math
import os
import re
import threading
import time
from typing import Any

import dash_leaflet as dl
import pandas as pd
import requests
from dash import (
    Dash,
    Input,
    Output,
    State,
    dash_table,
    dcc,
    html,
    no_update,
)

from dashboard_helpers import (
    NO_LOCATION_MESSAGE,
    TABLE_COLUMN_DEFINITIONS,
    VALID_RESCUE_TYPES,
    build_selected_row_styles,
    get_display_location,
    get_selected_candidate,
    get_table_dataframe,
    has_usable_location,
    prepare_dashboard_results,
    prepare_export_dataframe,
)

from run_full_matching import build_ranked_candidate_data


#############################
# Data Manipulation / Model
#############################

# Run the full pipeline once when the dashboard starts. Callbacks reuse
# these results instead of rebuilding more than 150,000 matches.
matched_records, ranked_candidates = build_ranked_candidate_data()

# Prepare the three rescue mission views and the combined All view.
dashboard_results = prepare_dashboard_results(
    ranked_candidates
)

# Start with all candidates and the strongest mission for each animal.
initial_dataframe = get_table_dataframe(
    dashboard_results,
    "all",
)


#########################
# Dashboard Configuration
#########################

app = Dash(__name__)

AUSTIN_CENTER = [
    30.2672,
    -97.7431,
]


PAGE_STYLE = {
    "minHeight": "100vh",
    "backgroundColor": "#f3f5f7",
    "fontFamily": (
        "system-ui, -apple-system, BlinkMacSystemFont, "
        "Segoe UI, Roboto, sans-serif"
    ),
    "color": "#1f2937",
}

CONTENT_STYLE = {
    "maxWidth": "1500px",
    "margin": "0 auto",
    "padding": "22px",
}

CARD_STYLE = {
    "backgroundColor": "white",
    "border": "1px solid #dfe3e8",
    "borderRadius": "12px",
    "boxShadow": "0 3px 12px rgba(0, 0, 0, 0.07)",
    "padding": "18px",
}

SECTION_TITLE_STYLE = {
    "margin": "0 0 16px 0",
    "fontSize": "22px",
    "color": "#263238",
}

BUTTON_STYLE = {
    "height": "42px",
    "padding": "0 18px",
    "border": "1px solid #9ca3af",
    "borderRadius": "7px",
    "backgroundColor": "white",
    "cursor": "pointer",
    "fontSize": "14px",
    "fontWeight": "600",
}

MISSION_LABEL_STYLE = {
    "display": "block",
    "padding": "11px 14px",
    "marginBottom": "10px",
    "border": "1px solid #c7cdd4",
    "borderRadius": "8px",
    "backgroundColor": "#f8fafc",
    "cursor": "pointer",
    "fontSize": "15px",
}

DETAIL_LABEL_STYLE = {
    "fontSize": "12px",
    "fontWeight": "700",
    "textTransform": "uppercase",
    "letterSpacing": "0.04em",
    "color": "#6b7280",
    "marginBottom": "3px",
}

DETAIL_VALUE_STYLE = {
    "fontSize": "15px",
    "fontWeight": "600",
    "color": "#1f2937",
    "overflowWrap": "anywhere",
}


#########################
# Logo
#########################

image_filename = (
    "Grazioso Salvare Logo.png"
)

if os.path.exists(image_filename):
    with open(
        image_filename,
        "rb",
    ) as image_file:
        encoded_image = base64.b64encode(
            image_file.read()
        ).decode("utf-8")
else:
    print(
        "Logo file not found:",
        image_filename,
    )
    encoded_image = None


#########################
# Location Helpers
#########################

# Successful results are cached for the current dashboard session.
# Failed searches are not cached because a temporary network issue
# should not cause an address to remain unavailable.
GEOCODE_CACHE: dict[
    str,
    tuple[float, float],
] = {}

GEOCODE_LOCK = threading.Lock()
LAST_GEOCODE_REQUEST_TIME = 0.0

GEOCODE_MINIMUM_INTERVAL = 1.05
GEOCODE_TIMEOUT = 8
GEOCODE_RETRY_COUNT = 2

GEOCODE_URL = (
    "https://nominatim.openstreetmap.org/search"
)

GEOCODE_HEADERS = {
    "User-Agent": (
        "Grazioso-Salvare-CS499-Dashboard/1.0 "
        "(educational capstone project)"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# This gives the geocoder a preference for the Austin and Travis
# County area without forcing every result to stay inside Austin.
CENTRAL_TEXAS_VIEWBOX = (
    "-98.35,30.75,-96.95,29.75"
)


def normalize_location(
    location: Any,
) -> str:
    """
    Convert shelter location text into a cleaner geocoding query.
    """

    if not has_usable_location(
        location
    ):
        return ""

    clean_location = str(
        location
    ).strip()

    # The archive commonly stores locations like:
    # "4601 Imperial Dr in Austin (TX)"
    # Change that into a format the geocoder understands better.
    clean_location = re.sub(
        r"\s+in\s+",
        ", ",
        clean_location,
        flags=re.IGNORECASE,
    )

    # The archive is not always consistent with the capitalization or
    # spacing around TX. Clean it up before building the search.
    clean_location = re.sub(
        r"\(\s*(?:TX|Texas)\s*\)",
        "TX",
        clean_location,
        flags=re.IGNORECASE,
    )

    clean_location = clean_location.replace(
        "&",
        " and ",
    )

    clean_location = re.sub(
        r"\s*,\s*",
        ", ",
        clean_location,
    )

    clean_location = re.sub(
        r"\s+",
        " ",
        clean_location,
    ).strip(" ,")

    if not re.search(
        r"\bTX\b|\bTexas\b",
        clean_location,
        flags=re.IGNORECASE,
    ):
        clean_location = (
            f"{clean_location}, Texas"
        )

    return clean_location


def expand_street_abbreviations(
    location: str,
) -> str:
    """
    Expand common street abbreviations for a fallback search.
    """

    replacements = {
        r"\bRd\b": "Road",
        r"\bDr\b": "Drive",
        r"\bAve\b": "Avenue",
        r"\bSt\b": "Street",
        r"\bLn\b": "Lane",
        r"\bBlvd\b": "Boulevard",
        r"\bCir\b": "Circle",
        r"\bCt\b": "Court",
        r"\bHwy\b": "Highway",
        r"\bFm\b": "Farm to Market Road",
        r"\bPkwy\b": "Parkway",
        r"\bPl\b": "Place",
        r"\bTrl\b": "Trail",
        r"\bCv\b": "Cove",
        r"\bTer\b": "Terrace",
    }

    expanded_location = location

    for pattern, replacement in replacements.items():
        expanded_location = re.sub(
            pattern,
            replacement,
            expanded_location,
            flags=re.IGNORECASE,
        )

    return expanded_location


def remove_texas_suffix(
    location: str,
) -> str:
    """
    Remove Texas from the end before adding a city or county.
    """

    return re.sub(
        r",?\s*(?:TX|Texas)\s*$",
        "",
        location,
        flags=re.IGNORECASE,
    ).strip(" ,")


def build_geocode_queries(
    location: Any,
) -> list[str]:
    """
    Build multiple reasonable search queries for one location.
    """

    normalized_location = normalize_location(
        location
    )

    if not normalized_location:
        return []

    expanded_location = expand_street_abbreviations(
        normalized_location
    )

    queries = [
        normalized_location,
    ]

    if (
        expanded_location.casefold()
        != normalized_location.casefold()
    ):
        queries.append(
            expanded_location
        )

    contains_street_number = bool(
        re.search(
            r"\b\d+\b",
            normalized_location,
        )
    )

    contains_known_area = bool(
        re.search(
            (
                r"\bAustin\b|\bTravis\b|\bDel Valle\b|"
                r"\bManor\b|\bPflugerville\b|\bLeander\b|"
                r"\bRound Rock\b|\bCedar Park\b|\bBuda\b|"
                r"\bKyle\b|\bBee Cave\b|\bLakeway\b|"
                r"\bBastrop\b"
            ),
            normalized_location,
            flags=re.IGNORECASE,
        )
    )

    # Some shelter records only contain a street address. Try Austin
    # and Travis County versions before giving up on the location.
    if (
        contains_street_number
        and not contains_known_area
    ):
        base_location = remove_texas_suffix(
            normalized_location
        )

        expanded_base_location = remove_texas_suffix(
            expanded_location
        )

        queries.append(
            f"{base_location}, Austin, Texas"
        )

        queries.append(
            (
                f"{expanded_base_location}, "
                "Travis County, Texas"
            )
        )

    # Do not send the same search twice. Keep the first readable
    # version if we end up building duplicate queries.
    unique_queries = []
    seen_queries = set()

    for query in queries:
        cleaned_query = re.sub(
            r"\s+",
            " ",
            query,
        ).strip(" ,")

        query_key = cleaned_query.casefold()

        if (
            cleaned_query
            and query_key not in seen_queries
        ):
            seen_queries.add(
                query_key
            )

            unique_queries.append(
                cleaned_query
            )

    return unique_queries


def wait_for_geocode_interval() -> None:
    """
    Enforce the public geocoder's request interval.
    """

    global LAST_GEOCODE_REQUEST_TIME

    elapsed_time = (
        time.monotonic()
        - LAST_GEOCODE_REQUEST_TIME
    )

    remaining_delay = (
        GEOCODE_MINIMUM_INTERVAL
        - elapsed_time
    )

    if remaining_delay > 0:
        time.sleep(
            remaining_delay
        )

    LAST_GEOCODE_REQUEST_TIME = (
        time.monotonic()
    )


def coordinates_are_valid(
    coordinates: Any,
) -> bool:
    """
    Check whether coordinates can safely be used by the map.
    """

    if not isinstance(
        coordinates,
        (tuple, list),
    ):
        return False

    if len(coordinates) != 2:
        return False

    try:
        latitude = float(
            coordinates[0]
        )

        longitude = float(
            coordinates[1]
        )

    except (
        TypeError,
        ValueError,
    ):
        return False

    # Make sure the values are inside real latitude and longitude
    # ranges before giving them to the map.
    return (
        -90 <= latitude <= 90
        and -180 <= longitude <= 180
        and latitude not in {
            float("inf"),
            float("-inf"),
        }
        and longitude not in {
            float("inf"),
            float("-inf"),
        }
        and latitude == latitude
        and longitude == longitude
    )


def request_geocode(
    query: str,
) -> tuple[float, float] | None:
    """
    Send one rate-limited request to the geocoding service.
    """

    for attempt in range(
        GEOCODE_RETRY_COUNT
    ):
        with GEOCODE_LOCK:
            wait_for_geocode_interval()

            try:
                response = requests.get(
                    GEOCODE_URL,
                    params={
                        "q": query,
                        "format": "jsonv2",
                        "limit": 1,
                        "countrycodes": "us",
                        "addressdetails": 1,
                        "viewbox": (
                            CENTRAL_TEXAS_VIEWBOX
                        ),
                        "bounded": 0,
                    },
                    headers=GEOCODE_HEADERS,
                    timeout=GEOCODE_TIMEOUT,
                )

                # The service may ask us to slow down. Use its delay
                # when one is provided instead of immediately retrying.
                if response.status_code == 429:
                    retry_after = response.headers.get(
                        "Retry-After"
                    )

                    if retry_after:
                        try:
                            time.sleep(
                                float(
                                    retry_after
                                )
                            )
                        except ValueError:
                            pass

                    if (
                        attempt
                        < GEOCODE_RETRY_COUNT - 1
                    ):
                        continue

                    return None

                # Try temporary server errors again before treating the
                # address as unavailable.
                if response.status_code in {
                    500,
                    502,
                    503,
                    504,
                }:
                    if (
                        attempt
                        < GEOCODE_RETRY_COUNT - 1
                    ):
                        continue

                    return None

                response.raise_for_status()

                results = response.json()

                if not results:
                    return None

                coordinates = (
                    float(
                        results[0]["lat"]
                    ),
                    float(
                        results[0]["lon"]
                    ),
                )

                # A response is only useful when both coordinate values
                # are valid and safe to pass to the map.
                if not coordinates_are_valid(
                    coordinates
                ):
                    return None

                return coordinates

            except (
                requests.RequestException,
                KeyError,
                TypeError,
                ValueError,
            ):
                if (
                    attempt
                    >= GEOCODE_RETRY_COUNT - 1
                ):
                    return None

    return None


def geocode_location(
    location: Any,
) -> tuple[float, float] | None:
    """
    Find approximate coordinates using cached and fallback searches.
    """

    normalized_location = normalize_location(
        location
    )

    if not normalized_location:
        return None

    cache_key = (
        normalized_location.casefold()
    )

    cached_coordinates = (
        GEOCODE_CACHE.get(
            cache_key
        )
    )

    if coordinates_are_valid(
        cached_coordinates
    ):
        return cached_coordinates

    queries = build_geocode_queries(
        location
    )

    for query in queries:
        coordinates = request_geocode(
            query
        )

        if coordinates_are_valid(
            coordinates
        ):
            # Only successful searches are cached. A failed request
            # should be allowed to try again on the next selection.
            GEOCODE_CACHE[
                cache_key
            ] = coordinates

            return coordinates

    return None


def is_specific_address(
    location: Any,
) -> bool:
    """
    Determine whether a location appears to contain a street address.
    """

    if not has_usable_location(
        location
    ):
        return False

    location_text = str(
        location
    )

    return bool(
        re.search(
            r"\b\d+\b",
            location_text,
        )
    )


def build_map(
    location: Any,
) -> html.Div:
    """
    Create the map for the selected candidate.
    """

    # Use the same missing location rules as the table and details.
    # Keep the original archive wording when it is available.
    location = get_display_location(
        location
    )

    coordinates = geocode_location(
        location
    )

    if not coordinates_are_valid(
        coordinates
    ):
        location_text = (
            NO_LOCATION_MESSAGE
        )

        map_center = AUSTIN_CENTER
        map_zoom = 10
        marker_children = []

    else:
        latitude, longitude = (
            coordinates
        )

        location_text = location

        map_center = [
            latitude,
            longitude,
        ]

        map_zoom = (
            15
            if is_specific_address(
                location
            )
            else 11
        )

        marker_children = [
            dl.Marker(
                position=map_center,
                children=[
                    dl.Tooltip(
                        location
                    ),
                    dl.Popup(
                        html.Div(
                            [
                                html.Strong(
                                    "Approximate location"
                                ),
                                html.Br(),
                                html.Span(
                                    location
                                ),
                            ]
                        )
                    ),
                ],
            )
        ]

    return html.Div(
        children=[
            html.Div(
                location_text,
                style={
                    "fontWeight": "600",
                    "marginBottom": "10px",
                    "minHeight": "22px",
                },
            ),
            dl.Map(
                center=map_center,
                zoom=map_zoom,
                style={
                    "width": "100%",
                    "height": "320px",
                    "borderRadius": "8px",
                },
                children=[
                    dl.TileLayer(),
                    *marker_children,
                ],
            ),
        ]
    )

#########################
# Candidate Detail Helpers
#########################


def display_value(
    value: Any,
    fallback: str = "Not available",
) -> str:
    """
    Return a clean value that can be shown in the dashboard.
    """

    if value is None:
        return fallback

    try:
        if bool(pd.isna(value)):
            return fallback
    except (
        TypeError,
        ValueError,
    ):
        pass

    clean_value = str(
        value
    ).strip()

    if clean_value.casefold() in {
        "",
        "<na>",
        "nan",
        "none",
    }:
        return fallback

    return clean_value


def display_number(
    value: Any,
    decimal_places: int = 0,
    fallback: str = "Not available",
) -> str:
    """
    Format a number without showing pandas missing values.
    """

    try:
        numeric_value = float(
            value
        )
    except (
        TypeError,
        ValueError,
    ):
        return fallback

    if not math.isfinite(
        numeric_value
    ):
        return fallback

    if decimal_places <= 0:
        return str(
            int(
                round(
                    numeric_value
                )
            )
        )

    return (
        f"{numeric_value:.{decimal_places}f}"
    )


def detail_item(
    label: str,
    value: Any,
) -> html.Div:
    """
    Build one label and value pair for the candidate details panel.
    """

    return html.Div(
        children=[
            html.Div(
                label,
                style=DETAIL_LABEL_STYLE,
            ),
            html.Div(
                display_value(
                    value
                ),
                style=DETAIL_VALUE_STYLE,
            ),
        ],
        style={
            "padding": "12px",
            "backgroundColor": "#f8fafc",
            "border": "1px solid #e5e7eb",
            "borderRadius": "8px",
            "minWidth": "0",
        },
    )


def score_item(
    label: str,
    value: Any,
    maximum_score: int,
) -> html.Div:
    """
    Build one score card for the selected candidate.
    """

    score_text = display_number(
        value
    )

    if score_text == "Not available":
        full_score_text = score_text
    else:
        full_score_text = (
            f"{score_text} / {maximum_score}"
        )

    return html.Div(
        children=[
            html.Div(
                label,
                style=DETAIL_LABEL_STYLE,
            ),
            html.Div(
                full_score_text,
                style={
                    **DETAIL_VALUE_STYLE,
                    "fontSize": "19px",
                    "color": "#1d4ed8",
                },
            ),
        ],
        style={
            "padding": "13px",
            "backgroundColor": "#eff6ff",
            "border": "1px solid #bfdbfe",
            "borderRadius": "8px",
            "minWidth": "0",
        },
    )


def build_candidate_details(
    candidate: dict[str, Any] | None,
) -> html.Div:
    """
    Build the detail panel for the selected candidate.
    """

    if not candidate:
        return html.Div(
            children=[
                html.Div(
                    "No candidate selected",
                    style={
                        "fontSize": "18px",
                        "fontWeight": "700",
                        "color": "#374151",
                        "marginBottom": "8px",
                    },
                ),
                html.Div(
                    (
                        "Select a candidate from the table to view "
                        "the rescue scoring details."
                    ),
                    style={
                        "color": "#6b7280",
                        "lineHeight": "1.5",
                    },
                ),
            ],
            style={
                "padding": "20px",
                "textAlign": "center",
            },
        )

    animal_id = display_value(
        candidate.get(
            "animal_id"
        )
    )

    candidate_name = display_value(
        candidate.get(
            "outcome_name"
        )
    )

    rescue_mission = display_value(
        candidate.get(
            "rescue_mission"
        )
    )

    total_score = display_number(
        candidate.get(
            "total_score"
        )
    )

    rescue_rank = display_number(
        candidate.get(
            "rescue_rank"
        )
    )

    # Put the most important candidate information at the top so the
    # user can quickly confirm which animal is currently selected.
    candidate_heading = html.Div(
        children=[
            html.Div(
                children=[
                    html.Div(
                        candidate_name,
                        style={
                            "fontSize": "25px",
                            "fontWeight": "750",
                            "color": "#1f2937",
                            "lineHeight": "1.2",
                        },
                    ),
                    html.Div(
                        f"Animal ID: {animal_id}",
                        style={
                            "marginTop": "5px",
                            "fontSize": "14px",
                            "color": "#6b7280",
                        },
                    ),
                ],
                style={
                    "minWidth": "0",
                },
            ),
            html.Div(
                children=[
                    html.Div(
                        rescue_mission,
                        style={
                            "fontSize": "15px",
                            "fontWeight": "700",
                            "color": "#1d4ed8",
                        },
                    ),
                    html.Div(
                        (
                            f"Rank {rescue_rank} | "
                            f"Total Score {total_score}"
                        ),
                        style={
                            "marginTop": "5px",
                            "fontSize": "14px",
                            "color": "#4b5563",
                        },
                    ),
                ],
                style={
                    "padding": "10px 14px",
                    "backgroundColor": "#eff6ff",
                    "border": "1px solid #bfdbfe",
                    "borderRadius": "8px",
                    "textAlign": "right",
                },
            ),
        ],
        style={
            "display": "flex",
            "justifyContent": "space-between",
            "alignItems": "flex-start",
            "gap": "18px",
            "flexWrap": "wrap",
            "marginBottom": "18px",
        },
    )

    candidate_information = html.Div(
        children=[
            detail_item(
                "Breed",
                candidate.get(
                    "outcome_breed"
                ),
            ),
            detail_item(
                "Age at Outcome",
                candidate.get(
                    "outcome_age_upon_outcome"
                ),
            ),
            detail_item(
                "Calculated Age in Weeks",
                display_number(
                    candidate.get(
                        "calculated_age_weeks"
                    )
                ),
            ),
            detail_item(
                "Sex Upon Outcome",
                candidate.get(
                    "outcome_sex_upon_outcome"
                ),
            ),
            detail_item(
                "Found Location",
                get_display_location(
                    candidate.get(
                        "found_location"
                    )
                ),
            ),
            detail_item(
                "Recommended Mission",
                candidate.get(
                    "rescue_mission"
                ),
            ),
        ],
        style={
            "display": "grid",
            "gridTemplateColumns": (
                "repeat(auto-fit, minmax(190px, 1fr))"
            ),
            "gap": "10px",
            "marginBottom": "18px",
        },
    )

    score_breakdown = html.Div(
        children=[
            score_item(
                "Total Score",
                candidate.get(
                    "total_score"
                ),
                100,
            ),
            score_item(
                "Breed Score",
                candidate.get(
                    "breed_score"
                ),
                60,
            ),
            score_item(
                "Age Score",
                candidate.get(
                    "age_score"
                ),
                25,
            ),
            score_item(
                "Sex Score",
                candidate.get(
                    "sex_score"
                ),
                15,
            ),
        ],
        style={
            "display": "grid",
            "gridTemplateColumns": (
                "repeat(auto-fit, minmax(150px, 1fr))"
            ),
            "gap": "10px",
            "marginBottom": "18px",
        },
    )

    # Keep the scoring explanation together so the user can see why
    # the candidate earned each part of the final score.
    score_explanation = html.Div(
        children=[
            html.Div(
                "Why This Candidate Ranked Here",
                style={
                    "fontSize": "17px",
                    "fontWeight": "700",
                    "color": "#263238",
                    "marginBottom": "12px",
                },
            ),
            html.Div(
                children=[
                    html.Div(
                        children=[
                            html.Strong(
                                "Breed: "
                            ),
                            html.Span(
                                display_value(
                                    candidate.get(
                                        "breed_reason"
                                    )
                                )
                            ),
                        ],
                        style={
                            "marginBottom": "9px",
                        },
                    ),
                    html.Div(
                        children=[
                            html.Strong(
                                "Age: "
                            ),
                            html.Span(
                                display_value(
                                    candidate.get(
                                        "age_reason"
                                    )
                                )
                            ),
                        ],
                        style={
                            "marginBottom": "9px",
                        },
                    ),
                    html.Div(
                        children=[
                            html.Strong(
                                "Sex: "
                            ),
                            html.Span(
                                display_value(
                                    candidate.get(
                                        "sex_reason"
                                    )
                                )
                            ),
                        ],
                        style={
                            "marginBottom": "9px",
                        },
                    ),
                    html.Div(
                        children=[
                            html.Strong(
                                "Overall: "
                            ),
                            html.Span(
                                display_value(
                                    candidate.get(
                                        "score_explanation"
                                    )
                                )
                            ),
                        ],
                    ),
                ],
                style={
                    "lineHeight": "1.55",
                    "color": "#374151",
                },
            ),
        ],
        style={
            "padding": "16px",
            "backgroundColor": "#f9fafb",
            "border": "1px solid #e5e7eb",
            "borderRadius": "8px",
        },
    )

    return html.Div(
        children=[
            candidate_heading,
            candidate_information,
            score_breakdown,
            score_explanation,
        ]
    )


#########################
# Dashboard Layout Helpers
#########################


def build_summary_card(
    title: str,
    value: Any,
    description: str,
) -> html.Div:
    """
    Build one summary card shown above the candidate table.
    """

    return html.Div(
        children=[
            html.Div(
                title,
                style={
                    "fontSize": "13px",
                    "fontWeight": "700",
                    "textTransform": "uppercase",
                    "letterSpacing": "0.04em",
                    "color": "#6b7280",
                },
            ),
            html.Div(
                display_value(
                    value
                ),
                style={
                    "marginTop": "7px",
                    "fontSize": "27px",
                    "fontWeight": "750",
                    "color": "#1f2937",
                },
            ),
            html.Div(
                description,
                style={
                    "marginTop": "5px",
                    "fontSize": "13px",
                    "color": "#6b7280",
                    "lineHeight": "1.4",
                },
            ),
        ],
        style={
            **CARD_STYLE,
            "padding": "16px",
        },
    )


def get_top_candidate_name(
    dataframe: pd.DataFrame,
) -> str:
    """
    Return the first candidate name from the current table view.
    """

    if dataframe.empty:
        return "None"

    if "outcome_name" not in dataframe.columns:
        return "Not available"

    return display_value(
        dataframe.iloc[0].get(
            "outcome_name"
        )
    )


def get_top_candidate_score(
    dataframe: pd.DataFrame,
) -> str:
    """
    Return the strongest score from the current table view.
    """

    if (
        dataframe.empty
        or "total_score"
        not in dataframe.columns
    ):
        return "Not available"

    return display_number(
        dataframe.iloc[0].get(
            "total_score"
        )
    )


def mission_title(
    rescue_type: str | None,
) -> str:
    """
    Return a readable title for the selected rescue mission.
    """

    mission_titles = {
        "all": "All Rescue Candidates",
        "water": "Water Rescue Candidates",
        "mountain": "Mountain Rescue Candidates",
        "disaster": "Disaster Rescue Candidates",
    }

    normalized_type = (
        rescue_type.strip().lower()
        if isinstance(
            rescue_type,
            str,
        )
        else "all"
    )

    return mission_titles.get(
        normalized_type,
        mission_titles["all"],
    )


#########################
# Dashboard Layout
#########################

initial_records = (
    initial_dataframe.to_dict(
        "records"
    )
)

initial_selected_rows = (
    [0]
    if initial_records
    else []
)

initial_candidate = get_selected_candidate(
    initial_records,
    initial_selected_rows,
)

initial_candidate_count = len(
    initial_dataframe
)

initial_top_candidate = (
    get_top_candidate_name(
        initial_dataframe
    )
)

initial_top_score = (
    get_top_candidate_score(
        initial_dataframe
    )
)


app.layout = html.Div(
    children=[
        dcc.Download(
            id="download-candidate-csv"
        ),

        html.Header(
            children=[
                html.Div(
                    children=[
                        (
                            html.Img(
                                src=(
                                    "data:image/png;base64,"
                                    f"{encoded_image}"
                                ),
                                style={
                                    "height": "90px",
                                    "width": "auto",
                                    "objectFit": "contain",
                                },
                            )
                            if encoded_image
                            else html.Div(
                                "Grazioso Salvare",
                                style={
                                    "fontSize": "28px",
                                    "fontWeight": "750",
                                },
                            )
                        ),
                        html.Div(
                            children=[
                                html.H1(
                                    (
                                        "Rescue Candidate "
                                        "Selection Dashboard"
                                    ),
                                    style={
                                        "margin": "0",
                                        "fontSize": "30px",
                                        "fontWeight": "750",
                                        "color": "white",
                                    },
                                ),
                                html.Div(
                                    (
                                        "Enhanced matching, scoring, "
                                        "ranking, and location support"
                                    ),
                                    style={
                                        "marginTop": "6px",
                                        "fontSize": "15px",
                                        "color": "#dbeafe",
                                    },
                                ),
                            ],
                        ),
                    ],
                    style={
                        "maxWidth": "1500px",
                        "margin": "0 auto",
                        "padding": "18px 22px",
                        "display": "flex",
                        "alignItems": "center",
                        "gap": "24px",
                    },
                ),
            ],
            style={
                "backgroundColor": "#1f4e79",
                "borderBottom": "5px solid #f4b942",
            },
        ),

        html.Main(
            children=[
                html.Div(
                    children=[
                        html.Div(
                            children=[
                                html.H2(
                                    "Rescue Mission",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                html.Div(
                                    (
                                        "Choose a mission to rank animals "
                                        "against that rescue profile."
                                    ),
                                    style={
                                        "marginBottom": "16px",
                                        "fontSize": "14px",
                                        "color": "#6b7280",
                                        "lineHeight": "1.5",
                                    },
                                ),
                                dcc.RadioItems(
                                    id="rescue-type-filter",
                                    options=[
                                        {
                                            "label": "All Candidates",
                                            "value": "all",
                                        },
                                        {
                                            "label": "Water Rescue",
                                            "value": "water",
                                        },
                                        {
                                            "label": "Mountain Rescue",
                                            "value": "mountain",
                                        },
                                        {
                                            "label": "Disaster Rescue",
                                            "value": "disaster",
                                        },
                                    ],
                                    value="all",
                                    labelStyle=MISSION_LABEL_STYLE,
                                    inputStyle={
                                        "marginRight": "9px",
                                    },
                                ),
                                html.Div(
                                    children=[
                                        html.Button(
                                            "Reset Dashboard",
                                            id="reset-dashboard-button",
                                            n_clicks=0,
                                            style={
                                                **BUTTON_STYLE,
                                                "width": "100%",
                                                "marginTop": "8px",
                                            },
                                        ),
                                        html.Button(
                                            "Export Current View",
                                            id="export-csv-button",
                                            n_clicks=0,
                                            style={
                                                **BUTTON_STYLE,
                                                "width": "100%",
                                                "marginTop": "10px",
                                                "backgroundColor": "#1f4e79",
                                                "color": "white",
                                                "borderColor": "#1f4e79",
                                            },
                                        ),
                                    ],
                                ),
                            ],
                            style=CARD_STYLE,
                        ),

                        html.Div(
                            children=[
                                html.H2(
                                    "Selected Candidate Location",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                html.Div(
                                    id="candidate-map",
                                    children=build_map(
                                        (
                                            initial_candidate.get(
                                                "found_location"
                                            )
                                            if initial_candidate
                                            else None
                                        )
                                    ),
                                ),
                                html.Div(
                                    (
                                        "Map results are approximate and "
                                        "depend on the location detail "
                                        "available in the shelter record."
                                    ),
                                    style={
                                        "marginTop": "12px",
                                        "fontSize": "12px",
                                        "color": "#6b7280",
                                        "lineHeight": "1.45",
                                    },
                                ),
                            ],
                            style={
                                **CARD_STYLE,
                                "marginTop": "18px",
                            },
                        ),
                    ],
                    style={
                        "minWidth": "280px",
                    },
                ),

                html.Div(
                    children=[
                        html.Div(
                            id="summary-cards",
                            children=[
                                build_summary_card(
                                    "Candidates",
                                    initial_candidate_count,
                                    (
                                        "Animals in the current "
                                        "rescue view"
                                    ),
                                ),
                                build_summary_card(
                                    "Top Candidate",
                                    initial_top_candidate,
                                    (
                                        "Highest ranked animal in "
                                        "the current view"
                                    ),
                                ),
                                build_summary_card(
                                    "Top Score",
                                    initial_top_score,
                                    (
                                        "Strongest suitability score "
                                        "out of 100"
                                    ),
                                ),
                            ],
                            style={
                                "display": "grid",
                                "gridTemplateColumns": (
                                    "repeat(3, minmax(0, 1fr))"
                                ),
                                "gap": "14px",
                                "marginBottom": "18px",
                            },
                        ),

                        html.Div(
                            children=[
                                html.Div(
                                    children=[
                                        html.Div(
                                            children=[
                                                html.H2(
                                                    id=(
                                                        "candidate-table-title"
                                                    ),
                                                    children=(
                                                        "All Rescue Candidates"
                                                    ),
                                                    style={
                                                        **SECTION_TITLE_STYLE,
                                                        "marginBottom": "4px",
                                                    },
                                                ),
                                                html.Div(
                                                    id=(
                                                        "candidate-table-summary"
                                                    ),
                                                    children=(
                                                        f"{initial_candidate_count} "
                                                        "candidates available"
                                                    ),
                                                    style={
                                                        "fontSize": "14px",
                                                        "color": "#6b7280",
                                                    },
                                                ),
                                            ],
                                        ),
                                    ],
                                    style={
                                        "marginBottom": "16px",
                                    },
                                ),

                                dash_table.DataTable(
                                    id="candidate-table",
                                    columns=(
                                        TABLE_COLUMN_DEFINITIONS
                                    ),
                                    data=initial_records,
                                    selected_rows=initial_selected_rows,
                                    page_current=0,
                                    page_size=10,
                                    page_action="native",
                                    sort_action="native",
                                    sort_mode="multi",
                                    filter_action="native",
                                    row_selectable="single",
                                    cell_selectable=False,
                                    fixed_rows={
                                        "headers": True
                                    },
                                    style_table={
                                        "overflowX": "auto",
                                        "maxHeight": "540px",
                                        "overflowY": "auto",
                                        "border": (
                                            "1px solid #dfe3e8"
                                        ),
                                        "borderRadius": "8px",
                                    },
                                    style_header={
                                        "backgroundColor": "#1f4e79",
                                        "color": "white",
                                        "fontWeight": "700",
                                        "border": (
                                            "1px solid #315f87"
                                        ),
                                        "padding": "11px",
                                        "textAlign": "left",
                                    },
                                    style_cell={
                                        "fontFamily": (
                                            "system-ui, -apple-system, "
                                            "BlinkMacSystemFont, Segoe UI, "
                                            "Roboto, sans-serif"
                                        ),
                                        "fontSize": "13px",
                                        "padding": "10px",
                                        "textAlign": "left",
                                        "whiteSpace": "normal",
                                        "height": "auto",
                                        "minWidth": "95px",
                                        "maxWidth": "260px",
                                        "overflow": "hidden",
                                        "textOverflow": "ellipsis",
                                        "border": (
                                            "1px solid #e5e7eb"
                                        ),
                                    },
                                    style_cell_conditional=[
                                        {
                                            "if": {
                                                "column_id": "rescue_rank"
                                            },
                                            "width": "65px",
                                            "minWidth": "65px",
                                            "maxWidth": "65px",
                                            "textAlign": "center",
                                            "fontWeight": "700",
                                        },
                                        {
                                            "if": {
                                                "column_id": "animal_id"
                                            },
                                            "width": "95px",
                                            "minWidth": "95px",
                                            "maxWidth": "95px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "rescue_mission"
                                                )
                                            },
                                            "width": "120px",
                                            "minWidth": "120px",
                                            "maxWidth": "120px",
                                        },
                                        {
                                            "if": {
                                                "column_id": "outcome_name"
                                            },
                                            "width": "120px",
                                            "minWidth": "120px",
                                            "maxWidth": "150px",
                                        },
                                        {
                                            "if": {
                                                "column_id": "outcome_breed"
                                            },
                                            "width": "190px",
                                            "minWidth": "160px",
                                            "maxWidth": "240px",
                                        },
                                        {
                                            "if": {
                                                "column_id": (
                                                    "outcome_age_upon_outcome"
                                                )
                                            },
                                            "width": "125px",
                                            "minWidth": "125px",
                                            "maxWidth": "145px",
                                        },
                                        {
                                            "if": {
                                                "column_id": "found_location"
                                            },
                                            "width": "260px",
                                            "minWidth": "220px",
                                            "maxWidth": "340px",
                                        },
                                    ],
                                    style_data_conditional=(
                                        build_selected_row_styles(
                                            initial_selected_rows
                                        )
                                    ),
                                    tooltip_data=[
                                        {
                                            column: {
                                                "value": display_value(
                                                    row.get(
                                                        column
                                                    )
                                                ),
                                                "type": "text",
                                            }
                                            for column in [
                                                "outcome_breed",
                                                "found_location",
                                            ]
                                        }
                                        for row in initial_records
                                    ],
                                    tooltip_duration=None,
                                ),
                            ],
                            style=CARD_STYLE,
                        ),

                        html.Div(
                            children=[
                                html.H2(
                                    "Selected Candidate Details",
                                    style=SECTION_TITLE_STYLE,
                                ),
                                html.Div(
                                    id="candidate-details",
                                    children=build_candidate_details(
                                        initial_candidate
                                    ),
                                ),
                            ],
                            style={
                                **CARD_STYLE,
                                "marginTop": "18px",
                            },
                        ),
                    ],
                    style={
                        "minWidth": "0",
                    },
                ),
            ],
            style={
                **CONTENT_STYLE,
                "display": "grid",
                "gridTemplateColumns": (
                    "minmax(280px, 330px) minmax(0, 1fr)"
                ),
                "gap": "18px",
                "alignItems": "start",
            },
        ),

        html.Footer(
            children=[
                html.Div(
                    (
                        "Grazioso Salvare Rescue Candidate Dashboard | "
                        "Dustin Davis | "
                        "CS 499 Computer Science Capstone"
                    ),
                    style={
                        "maxWidth": "1500px",
                        "margin": "0 auto",
                        "padding": "18px 22px",
                        "fontSize": "13px",
                        "color": "#6b7280",
                        "textAlign": "center",
                    },
                )
            ],
            style={
                "marginTop": "10px",
                "borderTop": "1px solid #dfe3e8",
                "backgroundColor": "white",
            },
        ),
    ],
    style=PAGE_STYLE,
)

#########################
# Dashboard Callbacks
#########################


@app.callback(
    Output(
        "candidate-table",
        "data",
    ),
    Output(
        "candidate-table",
        "selected_rows",
    ),
    Output(
        "candidate-table",
        "page_current",
    ),
    Output(
        "candidate-table-title",
        "children",
    ),
    Output(
        "candidate-table-summary",
        "children",
    ),
    Output(
        "summary-cards",
        "children",
    ),
    Output(
        "candidate-table",
        "tooltip_data",
    ),
    Input(
        "rescue-type-filter",
        "value",
    ),
)
def update_dashboard(
    rescue_type: str | None,
):
    """
    Update the candidate table when the rescue mission changes.
    """

    table_dataframe = get_table_dataframe(
        dashboard_results,
        rescue_type,
    )

    table_records = (
        table_dataframe.to_dict(
            "records"
        )
    )

    candidate_count = len(
        table_dataframe
    )

    top_candidate = (
        get_top_candidate_name(
            table_dataframe
        )
    )

    top_score = (
        get_top_candidate_score(
            table_dataframe
        )
    )

    table_title = mission_title(
        rescue_type
    )

    candidate_word = (
        "candidate"
        if candidate_count == 1
        else "candidates"
    )

    table_summary = (
        f"{candidate_count} "
        f"{candidate_word} available"
    )

    summary_cards = [
        build_summary_card(
            "Candidates",
            candidate_count,
            (
                "Animals in the current "
                "rescue view"
            ),
        ),
        build_summary_card(
            "Top Candidate",
            top_candidate,
            (
                "Highest ranked animal in "
                "the current view"
            ),
        ),
        build_summary_card(
            "Top Score",
            top_score,
            (
                "Strongest suitability score "
                "out of 100"
            ),
        ),
    ]

    # Start each mission view with the first candidate selected.
    selected_rows = (
        [0]
        if table_records
        else []
    )

    tooltip_data = [
        {
            column: {
                "value": display_value(
                    row.get(
                        column
                    )
                ),
                "type": "text",
            }
            for column in [
                "outcome_breed",
                "found_location",
            ]
        }
        for row in table_records
    ]

    return (
        table_records,
        selected_rows,
        0,
        table_title,
        table_summary,
        summary_cards,
        tooltip_data,
    )

@app.callback(
    Output(
        "candidate-details",
        "children",
    ),
    Output(
        "candidate-map",
        "children",
    ),
    Output(
        "candidate-table",
        "style_data_conditional",
    ),
    Input(
        "candidate-table",
        "derived_viewport_data",
    ),
    Input(
        "candidate-table",
        "selected_rows",
    ),
    State(
        "candidate-table",
        "data",
    ),
)
def update_selected_candidate(
    viewport_data: list[
        dict[str, Any]
    ] | None,
    selected_rows: list[int] | None,
    table_data: list[
        dict[str, Any]
    ] | None,
):
    """
    Update the details, map, and highlighted row.
    """

    # The viewport data follows the current page, sorting, and table
    # filters. Fall back to the full table during the initial load.
    current_view = (
        viewport_data
        if viewport_data is not None
        else table_data
    )

    candidate = get_selected_candidate(
        current_view,
        selected_rows,
    )

    candidate_details = (
        build_candidate_details(
            candidate
        )
    )

    if candidate:
        candidate_location = (
            candidate.get(
                "found_location"
            )
        )
    else:
        candidate_location = None

    candidate_map = build_map(
        candidate_location
    )

    selected_row_styles = (
        build_selected_row_styles(
            selected_rows
        )
    )

    return (
        candidate_details,
        candidate_map,
        selected_row_styles,
    )

@app.callback(
    Output(
        "download-candidate-csv",
        "data",
    ),
    Input(
        "export-csv-button",
        "n_clicks",
    ),
    State(
        "candidate-table",
        "derived_virtual_data",
    ),
    State(
        "candidate-table",
        "data",
    ),
    State(
        "rescue-type-filter",
        "value",
    ),
    prevent_initial_call=True,
)
def export_table_csv(
    n_clicks: int,
    virtual_data: list[
        dict[str, Any]
    ] | None,
    table_data: list[
        dict[str, Any]
    ] | None,
    rescue_type: str | None,
):
    """
    Export the current filtered and sorted candidate table.
    """

    if not n_clicks:
        return no_update

    # derived_virtual_data follows the filters and sorting currently
    # applied by the user. Fall back to the full mission table when
    # Dash has not created a virtual view yet.
    current_view = (
        virtual_data
        if virtual_data is not None
        else table_data
    )

    export_dataframe = (
        prepare_export_dataframe(
            current_view
        )
    )

    normalized_type = (
        rescue_type.strip().lower()
        if isinstance(
            rescue_type,
            str,
        )
        else "all"
    )

    if normalized_type not in (
        VALID_RESCUE_TYPES
    ):
        normalized_type = "all"

    export_filename = (
        "grazioso_salvare_"
        f"{normalized_type}_candidates.csv"
    )

    return dcc.send_data_frame(
        export_dataframe.to_csv,
        export_filename,
        index=False,
    )


@app.callback(
    Output(
        "rescue-type-filter",
        "value",
    ),
    Input(
        "reset-dashboard-button",
        "n_clicks",
    ),
    prevent_initial_call=True,
)
def reset_dashboard(
    n_clicks: int,
):
    """
    Return the dashboard to the complete candidate view.
    """

    if not n_clicks:
        return no_update

    # Changing the filter back to All also triggers the main dashboard
    # callback, which resets the table page and selected candidate.
    return "all"


#########################
# Run Dashboard
#########################

if __name__ == "__main__":
    app.run(
        debug=False,
        jupyter_mode="external",
    )

Dash app running on http://127.0.0.1:8050/
